# Jiao Swin-HSSAM — controlled fail-fast reproduction
Reproduksi terkontrol pada SNI 21-class instance crops. Tahap pertama hanya melatih Swin-T baseline (`SJ0`) dan model lengkap (`SJFULL`). Test tidak pernah dibuka. Checkpoint disimpan setiap epoch ke Google Drive dan Hugging Face agar dapat dipulihkan setelah reset atau pindah akun Colab.

In [ ]:
SEEDS = [42]
REPO_REF = 'agent/sni-instance-crops'
DRIVE_DATA_FOLDER = 'coffee-sni-instance-crop-v1'
DRIVE_RESULT_FOLDER = 'sni-jiao-hssam-v1'
HF_REPO = 'ediprin/coffee-backbone-checkpoints'
HF_NAMESPACE = 'sni-jiao-hssam-v1'
HF_SYNC_EVERY = 1

In [ ]:
# 1/5 SETUP REPOSITORY, DRIVE, GPU, DAN CHECKPOINT LINTAS AKUN
import csv, json, shutil, subprocess, sys, tarfile, time
import os
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
assert SEEDS == [42]
repo = Path('/content/coffee-bean-classification')
repo_url = 'https://github.com/ediprin/coffee-bean-classification.git'
def run(command, cwd=None):
    command = [str(item) for item in command]
    print('\n$', ' '.join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)
if (repo / '.git').is_dir():
    run(['git', 'fetch', 'origin', REPO_REF], cwd=repo)
    run(['git', 'checkout', REPO_REF], cwd=repo)
    run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=repo)
else:
    if repo.exists(): shutil.rmtree(repo)
    run(['git', 'clone', '--branch', REPO_REF, repo_url, repo])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', repo])
from huggingface_hub import HfApi, login
hf_token = userdata.get('HF_TOKEN')
assert hf_token, 'Tambahkan secret HF_TOKEN dengan izin write di Colab.'
login(token=hf_token, add_to_git_credential=False)
hf_user = HfApi().whoami(token=hf_token)['name']
os.environ['BILINEAR_LMMD_ARTIFACT_REPO'] = HF_REPO
os.environ['BILINEAR_LMMD_ARTIFACT_NAMESPACE'] = HF_NAMESPACE
os.environ['BILINEAR_LMMD_ARTIFACT_SYNC_EVERY'] = str(HF_SYNC_EVERY)
os.environ['BILINEAR_LMMD_ARTIFACT_REQUIRED'] = '1'
import torch
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('HF :', hf_user, '->', HF_REPO, '/', HF_NAMESPACE)
print('CHECKPOINT: wajib sinkron setiap', HF_SYNC_EVERY, 'epoch')

In [ ]:
# 2/5 PULIHKAN DATASET DARI SHARD GOOGLE DRIVE
data_root = Path('/content/sni-instance-crops')
drive_data = Path('/content/drive/MyDrive') / DRIVE_DATA_FOLDER
if not (data_root / 'audit.json').is_file():
    shards = sorted((drive_data / 'shards').glob('crop_shard_*.tar'))
    assert (drive_data / 'complete.json').is_file() and shards, 'Backup dataset belum lengkap.'
    data_root.mkdir(parents=True, exist_ok=True)
    for index, shard in enumerate(shards, 1):
        with tarfile.open(shard, 'r') as archive: archive.extractall(data_root, filter='data')
        if index % 5 == 0 or index == len(shards): print(f'RESTORE {index}/{len(shards)}', flush=True)
    for name in ('audit.json', 'manifest.csv'): shutil.copy2(drive_data / name, data_root / name)
    with (data_root / 'manifest.csv').open(newline='', encoding='utf-8') as handle:
        allowed = {row['crop_path'] for row in csv.DictReader(handle)}
    for path in (data_root / 'source').rglob('*.jpg'):
        if path.relative_to(data_root).as_posix() not in allowed: path.unlink()
audit = json.loads((data_root / 'audit.json').read_text())
assert audit['status'] == 'complete' and audit['num_classes'] == 21
assert audit['generated_cross_split_identity_count'] == 0 and audit['test_locked'] is True
output_root = Path('/content/drive/MyDrive') / DRIVE_RESULT_FOLDER
output_root.mkdir(parents=True, exist_ok=True)
print('DATASET:', data_root, '| CROPS:', audit['output_crops'])
print('OUTPUT:', output_root, '| TEST LOCKED:', audit['test_locked'])

In [ ]:
# 3/5 HELPER DENGAN HEARTBEAT 60 DETIK
def report(stage):
    path = output_root / f'val_reports/jiao_swin_hssam_{stage}.json'
    assert path.is_file(), f'Report belum ada: {path}'
    result = json.loads(path.read_text())
    print('\n=== PUTUSAN ===')
    for name, row in result['decisions'].items(): print(name, row['decision'], row['criteria'])
    print('FINAL:', result['final_decision'], '| TEST DIBUKA:', result['test_opened'])
    return result
def run_stage(stage, codes):
    command = [sys.executable, '-u', '-m', 'bilinear_lmmd.experiments.run_jiao_swin_hssam_screening', '--data-root', str(data_root), '--output-root', str(output_root), '--seeds', *map(str, SEEDS), '--stage', stage, '--evaluation-split', 'val', '--hf-repo', HF_REPO, '--hf-namespace', HF_NAMESPACE, '--hf-sync-every', str(HF_SYNC_EVERY)]
    process = subprocess.Popen(command, cwd=repo)
    started = time.monotonic()
    while process.poll() is None:
        status = []
        for code in codes:
            history = output_root / f'outputs/{code}_seed42/history.json'
            if history.is_file():
                try: status.append(f'{code}={len(json.loads(history.read_text()))}/50')
                except Exception: status.append(f'{code}=saving')
        print(f'[{stage.upper()} {(time.monotonic()-started)/60:.1f} menit]', ', '.join(status) if status else 'inisialisasi', flush=True)
        time.sleep(60)
    assert process.wait() == 0, 'Training gagal; baca traceback.'
    return report(stage)

In [ ]:
# 4/5 FAIL-FAST: SWIN-T BASELINE VS SWIN-HSSAM LENGKAP
screen = run_stage('screen', ('SJ0', 'SJFULL'))
if screen['final_decision'] != 'PASS': print('STOP. Jangan jalankan factorial atau seed lain.')

In [ ]:
# 5/5 OPSIONAL: TABLE-7 FACTORIAL, HANYA JIKA SCREEN PASS
screen = report('screen')
assert screen['final_decision'] == 'PASS', 'STOP: model lengkap tidak lolos.'
ablation = run_stage('ablation', ('SJH', 'SJS', 'SJL', 'SJHL', 'SJSL', 'SJHS'))